In [1]:
from src.utils.ingestion import file_loader,ppt_reader,preprocess


folder_path = "./knowledge_base/corpus/processed"
ppt_file_paths = file_loader.get_all_file_paths(folder_path)

c:\Users\AI ML PC Ajmeer\method-assesment-doc-search\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.utils.vectorstore import Milvus_init

milvus = Milvus_init()
print(milvus.initialize_collection(Drop_collection=True))

Collection ceated successfully


In [3]:

for index,path in enumerate(ppt_file_paths):
    print(f"file {index + 1} out of {len(ppt_file_paths)} ")
    data = ppt_reader.extract_unknow_slide(path)
    process_data = preprocess.preprocess_text(data,path)
    # print(process_data)
    milvus.milvus_insert_data(process_data)
    break

file 1 out of 7 
knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx
insert started
connection 1
connection 2
connection 3
data inserted successfully 


In [1]:
%pip install -q langchain_community 

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\AI ML PC Ajmeer\method-assesment-doc-search\myenv\Scripts\python.exe -m pip install --upgrade pip' command.


In [12]:
from dotenv import load_dotenv
import os
load_dotenv()

collection_name = os.getenv('MILVUS_COLLECTION_NAME')

In [25]:
from src.utils.embedding import embedder
e = embedder.Embedding()


In [26]:
from langchain_community.vectorstores import Milvus
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document
# from langchain_community.vectorstores.utils import has_mul_subcollection_support

from pymilvus import connections,Collection


def create_hybrid_retriever_milvus(documents: list[Document],collection_name: str = collection_name ,  k=4,
                                   embedding_model_name="sentence-transformers/LaBSE",
                                   host="localhost", port="19530"):

    # 1. Connect to Milvus
    connections.connect("default", host=host, port=port)

    
    milvusdb = Collection(collection_name)
    milvusdb.load()
    
   

    search_params = {
    "metric_type": "COSINE",
    "params": {"nprobe": 100},
}
 
    result = milvusdb.search([e.emb_text(d) for d in documents],"embeddings",search_params, limit=k, output_fields=["slide_title","isfullslide","slideno","embed_text","file_title","filename"])
  
    return result

    # return hybrid_retriever

print("done","--"*10)

done --------------------


In [27]:
a = ['We consider calibration with analysis of data, sensitivities, predictions, and uncertainty',
     "To what modeling project(s) do you plan to apply the methods?"]

result = create_hybrid_retriever_milvus(a,k=10)

In [28]:
for r in result:
    for e in r:
        print(e)

pk: 458483546193198846, distance: 1.0, entity: {'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'file_title': 'Geocenter, Copenhagen', 'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx'}
pk: 458483546193198849, distance: 0.6413075923919678, entity: {'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'Use sensitivity analysis to investigate relations between observations, parameters, and predictions.', 'file_title': 'Geocenter, Copenhagen', 'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx'}
pk: 458483546193198821, distance: 0.5729176998138428, entity: {'slide_title': 'Methods for sensitivity analysis, data asses

In [1]:
from src.utils.vectorstore import Milvus_init

m = Milvus_init()


c:\Users\AI ML PC Ajmeer\method-assesment-doc-search\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
a = ['We consider calibration with analysis of data, sensitivities, predictions, and uncertainty',
     "To what modeling project(s) do you plan to apply the methods?"]

# result = create_hybrid_retriever_milvus(a,k=10)
result = m.retriver(a[0],k=10)

In [6]:
import json
for r in result:
    for i in r:
        # json.dump(i)
        print(i)

pk: 458483546193198846, distance: 1.0, entity: {'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'file_title': 'Geocenter, Copenhagen', 'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx'}
pk: 458483546193198849, distance: 0.6413075923919678, entity: {'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'Use sensitivity analysis to investigate relations between observations, parameters, and predictions.', 'file_title': 'Geocenter, Copenhagen', 'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx'}
pk: 458483546193198821, distance: 0.5729176998138428, entity: {'slide_title': 'Methods for sensitivity analysis, data asses

In [13]:
import json
result[0]

["pk: 458483546193198846, distance: 1.0, entity: {'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx', 'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'file_title': 'Geocenter, Copenhagen'}", "pk: 458483546193198849, distance: 0.6413075923919678, entity: {'filename': 'knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx', 'slide_title': 'We consider calibration with analysis of data, sensitivities, predictions, and uncertainty', 'isfullslide': False, 'slideno': 11, 'embed_text': 'Use sensitivity analysis to investigate relations between observations, parameters, and predictions.', 'file_title': 'Geocenter, Copenhagen'}", "pk: 458483546193198821, distance: 0.5729176998138428, entity: {'filename': 'knowledge_base/corpus/processed/2BAXW45

In [19]:
from src.utils.embedding.embedder import Embedding

def retriver(
                query_text : str,
                 collection_name: str = None ,  
                 k=4):
        
        embed = Embedding()

        embed_query_text = embed.emb_text(query_text)

        if collection_name == None:
            # collection_name = self.COLLECTION_NAME
            collection_name = "ppt_docs"

        # 1. Connect to Milvus
        connections.connect("default", host= 'localhost', port=19530)

        
        milvusdb = Collection(collection_name)
        milvusdb.load()
        
    

        search_params = {
        "metric_type": "COSINE",
        "params": {"nprobe": 100},
    }
    
        result = milvusdb.search([embed_query_text,embed_query_text],"embeddings",search_params, limit=k, output_fields=["slide_title","isfullslide","slideno","embed_text","file_title","filename"])

        return result


In [36]:
import json
import re

def string_to_json(data_str):

    # Step 1: Convert the outer structure into a valid Python dictionary format
    # Add braces around the entire string
    data_str = "{" + data_str + "}"

    # Step 2: Fix keys that are not quoted using regex
    data_str = re.sub(r'(\b\w+\b):', r'"\1":', data_str)

    # Step 3: Replace single quotes with double quotes for JSON compatibility
    data_str = data_str.replace("'", '"')

    # Step 4: Fix Python-style booleans to JSON-style
    data_str = data_str.replace("False", "false").replace("True", "true")

    # Step 5: Parse as JSON
    data_dict = json.loads(data_str)

    # Optional: Pretty print the result
    json_str = json.dumps(data_dict, indent=4)

    return json_str

In [ ]:
import json

def string_to_json(data_str):
# Original input string

   



In [37]:
result = retriver(a[0],k=10)

In [47]:
for r in result:
    for i,j in enumerate(r):
        print(aka(str(j)))

{
    "slideno": 11,
    "embed_text": "We consider calibration with analysis of data, sensitivities, predictions, and uncertainty",
    "file_title": "Geocenter, Copenhagen",
    "filename": "knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx",
    "slide_title": "We consider calibration with analysis of data, sensitivities, predictions, and uncertainty",
    "isfullslide": false
}
{
    "slideno": 11,
    "embed_text": "Use sensitivity analysis to investigate relations between observations, parameters, and predictions.",
    "file_title": "Geocenter, Copenhagen",
    "filename": "knowledge_base/corpus/processed/2BAXW45Y275TQX4RSSQ7E2GPM6QNJQSL.pptx",
    "slide_title": "We consider calibration with analysis of data, sensitivities, predictions, and uncertainty",
    "isfullslide": false
}
{
    "slideno": 8,
    "embed_text": "Methods for sensitivity analysis, data assessment, model calibration, and evaluation of model uncertainty. These methods take advantage of th

In [ ]:
# for res in result:
#     for i in res:
#         json.dumps(i)
#         print("distance is" ,i['distance'])
#         for key,value in i['entity'].items():
#             print(f"{key} : {value}")
#         print('\n \n')

In [9]:
from langchain_community.vectorstores import Milvus
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document
from pymilvus import connections

def create_hybrid_retriever_from_precomputed(documents: list[Document], embeddings: list[list[float]], collection_name: str,
                                             k=4, host="localhost", port="19530"):

    # 1. Connect to Milvus
    connections.connect("default", host=host, port=port)

    # 2. Initialize embedding function (only needed for BM25 in this case)
    embedding_function = SentenceTransformerEmbeddings(model_name="sentence-transformers/LaBSE")

    # 3. Build Milvus vector store from texts + precomputed embeddings
    texts = [doc.page_content for doc in documents]
    metadatas = [doc.metadata for doc in documents]

    vector_store = Milvus.from_texts(
        texts=texts,
        embedding=embedding_function,  # Still needed by LangChain API but embeddings are passed
        metadatas=metadatas,
        collection_name=collection_name,
        embeddings=embeddings
    )

    # 4. Create retrievers
    milvus_retriever = vector_store.as_retriever(search_kwargs={"k": k})
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_retriever.k = k

    # 5. Combine both into an EnsembleRetriever
    hybrid_retriever = EnsembleRetriever(
        retrievers=[milvus_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )

    return hybrid_retriever


In [10]:
a = ['Calibration, Sensitivity Analysis and Uncertainty of Groundwater Models',
     "To what modeling project(s) do you plan to apply the methods?"]

print(create_hybrid_retriever_milvus(a,k=10))

AttributeError: type object 'Milvus' has no attribute 'load_local'